
A DVD rental company needs your help! They want to figure out how many days a customer will rent a DVD for based on some features and has approached you for help. They want you to try out some regression models which will help predict the number of days a customer will rent a DVD for. The company wants a model which yeilds a MSE of 3 or less on a test set. The model you make will help the company become more efficient inventory planning.

The data they provided is in the csv file `rental_info.csv`. It has the following features:
- `"rental_date"`: The date (and time) the customer rents the DVD.
- `"return_date"`: The date (and time) the customer returns the DVD.
- `"amount"`: The amount paid by the customer for renting the DVD.
- `"amount_2"`: The square of `"amount"`.
- `"rental_rate"`: The rate at which the DVD is rented for.
- `"rental_rate_2"`: The square of `"rental_rate"`.
- `"release_year"`: The year the movie being rented was released.
- `"length"`: Lenght of the movie being rented, in minuites.
- `"length_2"`: The square of `"length"`.
- `"replacement_cost"`: The amount it will cost the company to replace the DVD.
- `"special_features"`: Any special features, for example trailers/deleted scenes that the DVD also has.
- `"NC-17"`, `"PG"`, `"PG-13"`, `"R"`: These columns are dummy variables of the rating of the movie. It takes the value 1 if the move is rated as the column name and 0 otherwise. For your convinience, the reference dummy has already been dropped.

In [15]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

SEED = 9

In [2]:
# Load data
rental_df = pd.read_csv('datasets/rental_info.csv')

print(rental_df.head())

                 rental_date                return_date  amount  release_year  \
0  2005-05-25 02:54:33+00:00  2005-05-28 23:40:33+00:00    2.99        2005.0   
1  2005-06-15 23:19:16+00:00  2005-06-18 19:24:16+00:00    2.99        2005.0   
2  2005-07-10 04:27:45+00:00  2005-07-17 10:11:45+00:00    2.99        2005.0   
3  2005-07-31 12:06:41+00:00  2005-08-02 14:30:41+00:00    2.99        2005.0   
4  2005-08-19 12:30:04+00:00  2005-08-23 13:35:04+00:00    2.99        2005.0   

   rental_rate  length  replacement_cost                special_features  \
0         2.99   126.0             16.99  {Trailers,"Behind the Scenes"}   
1         2.99   126.0             16.99  {Trailers,"Behind the Scenes"}   
2         2.99   126.0             16.99  {Trailers,"Behind the Scenes"}   
3         2.99   126.0             16.99  {Trailers,"Behind the Scenes"}   
4         2.99   126.0             16.99  {Trailers,"Behind the Scenes"}   

   NC-17  PG  PG-13  R  amount_2  length_2  rental_rate_

In [5]:
# Check for missing values
print(rental_df.isnull().sum())

rental_date         0
return_date         0
amount              0
release_year        0
rental_rate         0
length              0
replacement_cost    0
special_features    0
NC-17               0
PG                  0
PG-13               0
R                   0
amount_2            0
length_2            0
rental_rate_2       0
dtype: int64


In [4]:
# Convert rental_date and return_date to datetime
rental_df['rental_date'] = pd.to_datetime(rental_df['rental_date'])
rental_df['return_date'] = pd.to_datetime(rental_df['return_date'])

# Check changes
print(rental_df.dtypes[['rental_date', 'return_date']])

rental_date    datetime64[us, UTC]
return_date    datetime64[us, UTC]
dtype: object


In [9]:
# Create rental_length_days column
rental_df['rental_length_days'] = (rental_df['return_date'] - rental_df['rental_date']).dt.days

# Check new column
print(rental_df[['rental_date', 'return_date', 'rental_length_days']].head())

                rental_date               return_date  rental_length_days
0 2005-05-25 02:54:33+00:00 2005-05-28 23:40:33+00:00                   3
1 2005-06-15 23:19:16+00:00 2005-06-18 19:24:16+00:00                   2
2 2005-07-10 04:27:45+00:00 2005-07-17 10:11:45+00:00                   7
3 2005-07-31 12:06:41+00:00 2005-08-02 14:30:41+00:00                   2
4 2005-08-19 12:30:04+00:00 2005-08-23 13:35:04+00:00                   4


In [11]:
# Check special_features column values
print(rental_df['special_features'].value_counts())

special_features
{Trailers,Commentaries,"Behind the Scenes"}                     1308
{Trailers}                                                      1139
{Trailers,Commentaries}                                         1129
{Trailers,"Behind the Scenes"}                                  1122
{"Behind the Scenes"}                                           1108
{Commentaries,"Deleted Scenes","Behind the Scenes"}             1101
{Commentaries}                                                  1089
{Commentaries,"Behind the Scenes"}                              1078
{Trailers,"Deleted Scenes"}                                     1047
{"Deleted Scenes","Behind the Scenes"}                          1035
{"Deleted Scenes"}                                              1023
{Commentaries,"Deleted Scenes"}                                 1011
{Trailers,Commentaries,"Deleted Scenes","Behind the Scenes"}     983
{Trailers,Commentaries,"Deleted Scenes"}                         916
{Trailers,"Delete

In [13]:
# Create dummy variables for special_features
rental_df["deleted_scenes"] = np.where(rental_df["special_features"].str.contains("Deleted Scenes"), 1, 0)
rental_df["behind_the_scenes"] = np.where(rental_df["special_features"].str.contains("Behind the Scenes"), 1, 0)

# Check new columns
print(rental_df[['deleted_scenes', 'behind_the_scenes']].head())

   deleted_scenes  behind_the_scenes
0               0                  1
1               0                  1
2               0                  1
3               0                  1
4               0                  1


In [14]:
# Create features and target variable
cols_to_drop = ['rental_date', 'return_date', 'special_features', 'rental_length_days']
X = rental_df.drop(columns=cols_to_drop)
y = rental_df['rental_length_days']

In [16]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

In [18]:
# Scale the features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [19]:
# Select features using lasso regression
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.3, random_state=SEED)
lasso.fit(X_train_scaled, y_train)

lasso_coefficients = lasso.coef_
selected_features = X.columns[lasso_coefficients != 0].to_list()

print("Selected features:", selected_features)

Selected features: ['amount', 'rental_rate', 'rental_rate_2']


In [20]:
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

In [ ]:
# Choosing models and performing hyperparameter tuning
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.metrics import root_mean_squared_error

In [22]:
# Baseline Models 
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score


models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=SEED, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(random_state=SEED),
    'Decision Tree': DecisionTreeRegressor(random_state=SEED)
}

baseline_results = {}

for model_name, model in models.items():
    cv_scores = cross_val_score(model, X_train_selected, y_train, cv=5, scoring='neg_root_mean_squared_error')
    baseline_results[model_name] = -cv_scores.mean()
    print(f"{model_name} - RMSE: {-cv_scores.mean():.4f}")

Linear Regression - RMSE: 1.7083
Random Forest - RMSE: 1.5514
Gradient Boosting - RMSE: 1.5539
Decision Tree - RMSE: 1.5514


In [23]:
print("Best baseline model:", min(baseline_results, key=baseline_results.get), "with RMSE:", min(baseline_results.values()))

Best baseline model: Random Forest with RMSE: 1.551419871248775


In [25]:
# Tune Random Forest Regressor
param_grid_rf = {
    'n_estimators': [50, 100, 150],
    'max_depth': [6, 10, 15, None],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', None]
}

grid_search_rf = GridSearchCV(RandomForestRegressor(random_state=SEED), 
                            param_grid_rf, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)

grid_search_rf.fit(X_train_selected, y_train)

print("Best parameters for Random Forest:", grid_search_rf.best_params_)
print("Best RMSE for Random Forest:", -grid_search_rf.best_score_)

Best parameters for Random Forest: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 100}
Best RMSE for Random Forest: 1.551419871248775


In [26]:
# Tuned model is the same with the baseline, thus try gradient boosting regressor
param_grid_gb = {
    'n_estimators': [100, 150, 200],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [2, 3, 4],
    'subsample': [0.8, 1.0],
    'max_features': ['sqrt', None]
}

grid_search_gb = GridSearchCV(GradientBoostingRegressor(random_state=SEED),
                            param_grid_gb, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)

grid_search_gb.fit(X_train_selected, y_train)
print("Best parameters for Gradient Boosting:", grid_search_gb.best_params_)
print("Best RMSE for Gradient Boosting:", -grid_search_gb.best_score_)

Best parameters for Gradient Boosting: {'learning_rate': 0.1, 'max_depth': 3, 'max_features': 'sqrt', 'n_estimators': 200, 'subsample': 0.8}
Best RMSE for Gradient Boosting: 1.5513509553779283


In [27]:
# Slightly better than tuned random forest
best_model = grid_search_gb.best_estimator_
best_mse = mean_squared_error(y_test, best_model.predict(X_test_selected))
print("Best model test MSE:", best_mse)

Best model test MSE: 2.503005076941551
